# clipsmith — Local VOD + Ollama (free, T4 GPU)

Download a Twitch VOD and process it entirely for free using Ollama as the LLM and the T4 GPU for fast transcription.

**What this notebook does:**
1. Installs clipsmith + GPU transcription dependencies
2. Installs the Ollama server and pulls a model (~4.7 GB, once per session)
3. Downloads the VOD from Twitch
4. Transcribes with faster-whisper on the T4 GPU (~8–12 min for a 2-hr VOD)
5. Selects the best moments with Ollama — no API key or cost
6. Cuts final clips with ffmpeg

**Requirements:**
- Runtime: **GPU → T4** (Runtime → Change runtime type → T4 GPU)
- `TWITCH_CLIENT_ID` + `TWITCH_CLIENT_SECRET` in Colab Secrets (optional but recommended)
- No LLM API key needed

In [ ]:
# @title 1. Install clipsmith + GPU transcription dependencies
!pip install -q "clipsmith-ai[ollama]"
# GPU-capable faster-whisper (CUDA 11.x bundled with T4 Colab runtime)
!pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11
!ffmpeg -version | head -1
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
)
print(f"GPU: {result.stdout.strip() or 'NOT FOUND — switch to T4 runtime'}")

In [ ]:
# @title 2. Load Twitch credentials from Colab Secrets (optional)
import os
from google.colab import userdata

os.environ["TWITCH_CLIENT_ID"] = userdata.get("TWITCH_CLIENT_ID") or ""
os.environ["TWITCH_CLIENT_SECRET"] = userdata.get("TWITCH_CLIENT_SECRET") or ""

if not os.environ["TWITCH_CLIENT_ID"]:
    print("Note: Twitch credentials not set — existing-clip boost signals will be skipped")
else:
    print("Twitch credentials loaded.")

In [ ]:
# @title 3. Install Ollama server and pull model
# Pulling ~4.7 GB takes ~3–5 min on first run per session.
# Uncomment the Drive lines to persist and skip re-download on future sessions.
import subprocess
import threading
import time

OLLAMA_MODEL = "llama3.1:8b"  # alternatives: mistral:7b (~4.1 GB), phi3:mini (~2.3 GB)

# --- Optional: persist model on Drive ---
# from google.colab import drive
# drive.mount("/content/drive")
# import os; os.makedirs("/content/drive/MyDrive/ollama_models", exist_ok=True)
# os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
# ----------------------------------------

# pciutils: GPU detection; zstd: required by the Ollama installer for extraction
get_ipython().system("apt-get install -y -q pciutils zstd")
get_ipython().system("curl -fsSL https://ollama.com/install.sh | sh")


# Start ollama serve in a background thread so it doesn't block the cell
def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])


thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)  # wait for server to be ready

get_ipython().system(f"ollama pull {OLLAMA_MODEL}")
print("Ollama ready.")

In [ ]:
# @title 4. Configure — set your VOD ID here
from pathlib import Path

VOD_ID = "2758856582"  # <-- paste your Twitch VOD ID
LANGUAGE = "es"  # ISO 639-1 — change to 'en' for English streams
WHISPER_MODEL = "medium"  # medium is fast + accurate on T4; large-v3 for best quality
MAX_CANDIDATES = 20

config_yaml = f"""\
work_dir: /content/work
out_dir:  /content/out

llm:
  provider: ollama
  model_ollama: {OLLAMA_MODEL}

transcribe:
  model: {WHISPER_MODEL}
  compute_type: float16  # uses GPU automatically when available; change to int8 for CPU
  language: {LANGUAGE}
  chunk_minutes: 10
  max_workers: 2

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("/content/work").mkdir(parents=True, exist_ok=True)
Path("/content/out").mkdir(parents=True, exist_ok=True)
Path("config.yaml").write_text(config_yaml)

# Remove cached transcript so it re-runs with the correct device
cached = Path(f"/content/work/{VOD_ID}/transcript.json")
if cached.exists():
    cached.unlink()
    print("Cleared cached transcript — will re-transcribe on GPU.")

print(f"Config written — VOD: {VOD_ID}, Whisper: {WHISPER_MODEL}, LLM: ollama/{OLLAMA_MODEL}")

In [ ]:
# @title 5. Run the full pipeline (download → transcribe → select → cut)
import sys

cmd = " ".join(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "run-vod",
        VOD_ID,
        "--config",
        "config.yaml",
        "--max-candidates",
        str(MAX_CANDIDATES),
        "-v",
    ]
)
ret = get_ipython().system(cmd)
if ret:
    raise RuntimeError(f"clipsmith exited with code {ret}")

In [ ]:
# @title 6. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clips = sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4"))

if not clips:
    print("No clips found. Did step 5 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))

In [ ]:
# @title 7. Download clips to your computer
from google.colab import files
from pathlib import Path

for mp4 in sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4")):
    files.download(str(mp4))